# TBot_SA1 策略学习 Notebook

本 notebook 用于理解 **TBot_SA1** 的完整数据流：从 LeRobot 数据集原始字段 → `DataTransformFn` 预处理链 → `forward` / `select_action` 推理。

> **环境要求**：`conda activate tbot_sa1`，并确保 `PYTHONPATH` 包含 `/vla/my_tbot/src`。
>
> **与 PI05 官方模式的差异**：TBot 不使用 `make_pre_post_processors`，而是在 `TBotSA1DatasetConfig.data_transforms` 中定义变换链，由 `TransformedLeRobotDataset` 在 `__getitem__` 时自动执行。

In [1]:
import os
import sys
from pathlib import Path

# 离线模式：只用本地模型/数据集，禁止 HuggingFace 联网
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

WORKSPACE_ROOT = Path("/vla/workspace/my_tbot")
SRC_ROOT = WORKSPACE_ROOT / "src"
MODELS_ROOT = Path("/vla/.models")
DATA_ROOT = Path("/vla/workspace/data")

# 若未 pip install -e 当前仓库，需手动加入 src
src_root_str = str(SRC_ROOT)
if src_root_str not in sys.path:
    sys.path.insert(0, src_root_str)

## 1. 模型加载

TBot 策略类为 `TBotSA1Policy`，checkpoint 目录需包含 `config.json` 和 `model.safetensors`。

**config 中说明了输入/输出字段**（state/action 会被 pad 到 `max_state_dim=32` / `max_action_dim=32`）。

In [2]:
import torch
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.TBot_SA1.modeling_tbot_sa1 import TBotSA1Policy

# ---------- 路径配置（按需修改）----------
MODEL_ID = "/vla/workspace/models/tbot_base"
QWEN3_VL_PATH = "/vla/workspace/models/Qwen3-VL-2B-Instruct"
COSMOS_PATH = "/vla/workspace/models/Cosmos-Tokenizer-CI8x8"
DA3_PATH = "/vla/workspace/models/DA3-LARGE-1.1"
DA3_CODE_ROOT = "/vla/workspace/my_tbot/third_party/Depth-Anything-3"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 从 checkpoint 读取 config，并覆盖为当前机器上的本地路径
policy_cfg = PreTrainedConfig.from_pretrained(MODEL_ID, local_files_only=True)
policy_cfg.qwen3_vl_pretrained_path = QWEN3_VL_PATH
policy_cfg.cosmos_tokenizer_path_or_name = COSMOS_PATH
policy_cfg.da3_model_path_or_name = DA3_PATH
policy_cfg.da3_code_root = DA3_CODE_ROOT
policy_cfg.device = str(device)

policy = TBotSA1Policy.from_pretrained(
    MODEL_ID,
    config=policy_cfg,
    local_files_only=True,
).to(device).eval()

policy

/vla/.conda/miniconda3/envs/mytbot/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
[INFO ] using MLP layer as FFN
Loading weights from local directory
Loading weights from local directory


TBotSA1Policy(
  (model): TBotSA1Model(
    (qwen3_vl_with_expert): Qwen3VLWithExpertModel(
      (und_expert): Qwen3VLForConditionalGeneration(
        (model): Qwen3VLModel(
          (visual): Qwen3VLVisionModel(
            (patch_embed): Qwen3VLVisionPatchEmbed(
              (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
            )
            (pos_embed): Embedding(2304, 1024)
            (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
            (blocks): ModuleList(
              (0-23): 24 x Qwen3VLVisionBlock(
                (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
                (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
                (attn): Qwen3VLVisionAttention(
                  (qkv): Linear(in_features=1024, out_features=3072, bias=True)
                  (proj): Linear(in_features=1024, out_features=1024, bias=True)
                )
                (mlp): Qwen3VLVisionMLP(
                  

In [3]:
print("=== input_features ===")
for k, v in policy.config.input_features.items():
    print(f"  {k}: {v}")

print("\n=== output_features ===")
for k, v in policy.config.output_features.items():
    print(f"  {k}: {v}")

print("\n=== 关键超参 ===")
print(f"chunk_size={policy.config.chunk_size}, n_action_steps={policy.config.n_action_steps}")
print(f"image_delta_indices={policy.config.image_delta_indices}")
print(f"dtype={policy.config.dtype}, num_inference_steps={policy.config.num_inference_steps}")

=== input_features ===
  observation.state: PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))

=== output_features ===
  action: PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))

=== 关键超参 ===
chunk_size=50, n_action_steps=50
image_delta_indices=[-15, 0, 15]
dtype=bfloat16, num_inference_steps=10


In [4]:
policy.config

TBotSA1Config(n_obs_steps=1, input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda', use_amp=False, push_to_hub=False, repo_id='zaleni/TBot-SA1-Base', private=None, tags=None, license=None, pretrained_path=None, qwen3_vl_variant='qwen3_vl_28l', action_expert_variant='qwen3_28l', qwen3_vl_pretrained_path='/vla/workspace/models/Qwen3-VL-2B-Instruct', dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, mask_action_dim_padding_loss=False, action_loss_valid_dim=None, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampling_offset=0.001, min_period=0.004, max_period=4.0, attention_mask_mode='default', image_resolution=(224, 224), image_delta_indices=[-15, 0, 15], empty_cameras=0, normalization_mapping={'VISUAL': <NormalizationMode.IDENTITY

TBotSA1Config(n_obs_steps=1, 
input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, 
output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda', use_amp=False, push_to_hub=False, repo_id='zaleni/TBot-SA1-Base', private=None, tags=None, license=None, pretrained_path=None, 

qwen3_vl_variant='qwen3_vl_28l', action_expert_variant='qwen3_28l',
qwen3_vl_pretrained_path='/vla/.models/Qwen3-VL-2B-Instruct', 

dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, mask_action_dim_padding_loss=False, action_loss_valid_dim=None, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampling_offset=0.001, min_period=0.004, max_period=4.0, attention_mask_mode='default', image_resolution=(224, 224), image_delta_indices=[-15, 0, 15], empty_cameras=0, normalization_mapping={'VISUAL': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'STATE': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'ACTION': <NormalizationMode.IDENTITY: 'IDENTITY'>}, gradient_checkpointing=False, compile_model=False, compile_mode='max-autotune', optimizer_lr=5e-05, optimizer_betas=(0.9, 0.95), optimizer_eps=1e-08, optimizer_weight_decay=0.01, optimizer_grad_clip_norm=1.0, scheduler_warmup_steps=2000, scheduler_decay_steps=300000, scheduler_decay_lr=1e-05, tokenizer_max_length=48, freeze_vision_encoder=False, train_expert_only=False, train_vlm_only=False, lora_modules=(), lora_unselected_mode='full', lora_targets=('attn', 'ffn'), lora_rank=16, lora_alpha=32.0, lora_rank_und=None, lora_alpha_und=None, lora_rank_gen=None, lora_alpha_gen=None, lora_rank_act=None, lora_alpha_act=None, lora_dropout=0.0, scale_factor=8, lambda_gen=0.01, cosmos_tokenizer_path_or_name='/vla/.models/Cosmos-Tokenizer-CI8x8', enable_3d_queries=True, num_3d_query_tokens=432, da3_alignment_mode='query_decoder', da3_query_resampler_layers=1, da3_query_resampler_ff_mult=1, query_layer_indices=(13, 19, 23, 27), da3_variant='auto', da3_teacher_layers=(11, 15, 19, 23), da3_query_dim=2048, da3_tokens_per_view=1296, da3_num_views=3, lambda_3d=0.01, da3_model_path_or_name='/vla/.models/DA3-LARGE-1.1', da3_model_name=None, da3_code_root='/vla/my_tbot/third_party/Depth-Anything-3', da3_teacher_process_res=504, da3_layer_weights=(1.0, 1.2, 1.4, 1.6), future_query_init_std=0.02, log_da3_teacher_timing=True)

## 2. 数据集处理

TBot 训练使用 `TransformedLeRobotDataset`：底层 `LeRobotDataset` 负责按 `delta_timestamps` 取时序帧，外层 transform 链负责归一化、图像重映射、Qwen3-VL tokenization 等。

In [5]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_PATH = "/vla/workspace/data/adjust_bottle/aloha-agilex_clean_50"  # <- 按需替换

# 原始数据集（未经 transform）
raw_ds = LeRobotDataset(
    repo_id=DATASET_PATH,
    video_backend="pyav",
)
raw_ds

LeRobotDataset({
    Repository ID: '/vla/workspace/data/adjust_bottle/aloha-agilex_clean_50',
    Number of selected episodes: '50',
    Number of selected samples: '7188',
    Features: '['observation.state', 'action', 'observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [6]:
# 原始字段与 shape
for k in [
    "action",
    "observation.state",
    "observation.images.cam_high",
    "observation.images.cam_left_wrist",
    "observation.images.cam_right_wrist",
]:
    if k in raw_ds[0]:
        v = raw_ds[0][k]
        print(f"{k:45s} {tuple(v.shape) if hasattr(v, 'shape') else type(v)} {getattr(v, 'dtype', '')}")
    else:
        print(f"{k:45s} (not in dataset)")
print("task:", raw_ds[0].get("task"))

  warnings.warn(



action                                        (14,) torch.float32
observation.state                             (14,) torch.float32
observation.images.cam_high                   (3, 480, 640) torch.float32
observation.images.cam_left_wrist             (3, 480, 640) torch.float32
observation.images.cam_right_wrist            (3, 480, 640) torch.float32
task: Lift the bottle with narrow top head-up from the table


### 2.1 构建与训练一致的数据集

`TBotSA1DatasetConfig.data_transforms.inputs` 定义了完整预处理链（见下一节逐步拆解）：

1. `DeltaActionTransformFn` — 可选，将 action 转为相对 state 的 delta
2. `ResizeImagesWithPadFn` — 图像 resize+pad 到 224×224
3. `RemapImageKeyTransformFn` — 相机键名 → `observation.images.image{0,1,2}`
4. `NormalizeTransformFn` — state/action 归一化（从 dataset.meta.stats 注入）
5. `ComposeFieldsTransform` — 合并多字段 state/action
6. `PadStateAndActionTransformFn` — pad 到 max_state_dim / max_action_dim
7. `Qwen3_VLProcessorTransformFn` — 图像 token + 语言 token
8. `UnifyTBotSA1InputsTransformFn` — 整理为 policy 期望的键集合

同时 `resolve_delta_timestamps` 会为 action 取 `chunk_size` 步、为每路相机取 `image_delta_indices=[-15,0,15]` 三帧。

In [7]:
from lerobot.configs.train import TrainPipelineConfig
from lerobot.datasets.factory import _build_single_dataset
from lerobot.policies.TBot_SA1.configuration_tbot_sa1 import TBotSA1DatasetConfig

dataset_cfg = TBotSA1DatasetConfig(
    repo_id=DATASET_PATH,
    qwen3_vl_processor_path=QWEN3_VL_PATH,
    video_backend="pyav",
)

train_cfg = TrainPipelineConfig(dataset=dataset_cfg, policy=policy_cfg, batch_size=1)

# 与 lerobot_train.py 中 make_dataset → _build_single_dataset 一致
transformed_ds, dataset_stats, robot_type = _build_single_dataset(
    train_cfg,
    DATASET_PATH,
    image_transforms=None,
    seed_offset=0,
)

print("robot_type:", robot_type)
print("transform pipeline:")
for i, step in enumerate(dataset_cfg.data_transforms.inputs):
    print(f"  [{i}] {step.__class__.__name__}")

transformed_ds

Hydrating transform InjectMissingStateActionTransformFn (robot_type=aloha, resolved=aloha, action_seq_len=1, state_seq_len=1, placeholder_dim=14)
Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)
robot_type: aloha
transform pipeline:
  [0] InjectMissingStateActionTransformFn
  [1] ResizeImagesWithPadFn
  [2] RemapImageKeyTransformFn
  [3] NormalizeTransformFn
  [4] ComposeFieldsTransform
  [5] PadStateAndActionTransformFn
  [6] Qwen3_VLProcessorTransformFn
  [7] UnifyTBotSA1InputsTransformFn


TransformedLeRobotDataset({
    Repository ID: '/vla/workspace/data/adjust_bottle/aloha-agilex_clean_50',
    Number of selected episodes: '50',
    Number of selected samples: '7188',
    Features: '['observation.state', 'action', 'observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})', (Transformed from LeRobotDataset)

注意 这里的图像数据是3帧

### 2.2 逐步观察 transform（类似 PI05 的 preprocess.steps）

下面手动逐步执行 transform，观察键名与张量 shape 的变化。

In [8]:
from dataclasses import replace
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.transforms.core import (
    compose,
    hydrate_normalize_transform,
    hydrate_compose_field_transform,
    hydrate_delta_action_transform,
    hydrate_remap_image_key_transform,
)

# 带 delta_timestamps 的原始帧（与 TransformedLeRobotDataset 底层一致）
ds_meta = LeRobotDatasetMetadata(DATASET_PATH)
delta_ts = resolve_delta_timestamps(policy_cfg, ds_meta)

base_ds = LeRobotDataset(
    repo_id=DATASET_PATH,
    video_backend="pyav",
    delta_timestamps=delta_ts,
)
frame = dict(base_ds[0])
print("原始 frame keys:", sorted(frame.keys()))
print("action shape:", tuple(frame["action"].shape))
print("observation.state shape:", tuple(frame["observation.state"].shape))

# hydrate 后的 transform 链
steps = list(dataset_cfg.data_transforms.inputs)
steps = hydrate_normalize_transform(steps, base_ds)
steps = hydrate_compose_field_transform(steps, base_ds)
steps = hydrate_delta_action_transform(steps, base_ds)
steps = hydrate_remap_image_key_transform(steps, base_ds)

data = frame
for i, step in enumerate(steps):
    data = step(data)
    tensor_shapes = {
        k: tuple(v.shape)
        for k, v in data.items()
        if hasattr(v, "shape")
    }
    print(f"\n--- step {i}: {step.__class__.__name__} ---")
    print("keys:", sorted(data.keys()))
    for k, shape in sorted(tensor_shapes.items()):
        print(f"  {k}: {shape}")

  warnings.warn(



原始 frame keys: ['action', 'action_is_pad', 'episode_index', 'frame_index', 'index', 'observation.images.cam_high', 'observation.images.cam_high_is_pad', 'observation.images.cam_left_wrist', 'observation.images.cam_left_wrist_is_pad', 'observation.images.cam_right_wrist', 'observation.images.cam_right_wrist_is_pad', 'observation.state', 'robot_type', 'task', 'task_index', 'timestamp']
action shape: (50, 14)
observation.state shape: (14,)
Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)

--- step 0: InjectMissingStateActionTransformFn ---
keys: ['action', 'action_is_pad', 'episode_index', 'frame_index', 'index', 'observation.images.cam_high', 'observation.images.cam_high_is_pad', 'observation

In [9]:
steps

[InjectMissingStateActionTransformFn(mapping={}, robot_types_without_action=('egodex_v',), action_seq_len=1, state_seq_len=1, placeholder_dim=1),
 ResizeImagesWithPadFn(height=224, width=224, mode='bilinear'),
 RemapImageKeyTransformFn(mapping={'observation.images.cam_high': 'observation.images.image0', 'observation.images.cam_left_wrist': 'observation.images.image1', 'observation.images.cam_right_wrist': 'observation.images.image2'}),
 NormalizeTransformFn(selected_keys=['observation.state', 'action'], mode='mean_std', norm_stats={'observation.images.cam_high': {'min': array([[[0.]],
 
        [[0.]],
 
        [[0.]]]), 'max': array([[[1.]],
 
        [[1.]],
 
        [[1.]]]), 'mean': array([[[0.8465936 ]],
 
        [[0.83313183]],
 
        [[0.83217112]]]), 'std': array([[[0.2597468 ]],
 
        [[0.25618517]],
 
        [[0.25913026]]]), 'count': array([5000])}, 'timestamp': {'min': array([0.]), 'max': array([5.73333333]), 'mean': array([2.3844092]), 'std': array([1.39251337])

In [10]:
# 完整 transform 后的单样本（与 transformed_ds[0] 等价）
sample = transformed_ds[0]
print("预处理后 keys:", sorted(sample.keys()))
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:40s} {tuple(v.shape)}")
    else:
        print(f"  {k:40s} {v}")

预处理后 keys: ['action', 'observation.attention_mask', 'observation.image_grid_thw', 'observation.images.image0', 'observation.images.image0_mask', 'observation.images.image1', 'observation.images.image1_mask', 'observation.images.image2', 'observation.images.image2_mask', 'observation.input_ids', 'observation.pixel_values', 'observation.state', 'sample.action_loss_mask']
  observation.state                        (32,)
  action                                   (50, 32)
  sample.action_loss_mask                  (1,)
  observation.images.image0                (3, 3, 224, 224)
  observation.images.image1                (3, 3, 224, 224)
  observation.images.image2                (3, 3, 224, 224)
  observation.images.image0_mask           ()
  observation.images.image1_mask           ()
  observation.images.image2_mask           ()
  observation.pixel_values                 (768, 1536)
  observation.image_grid_thw               (3, 3)
  observation.input_ids                    (246,)
  obse

### 2.3 组装 batch

训练时 `DataLoader` 默认 collate 会把样本 stack 成 batch。推理时 **不需要** `action` 输入，但 `forward` 训练需要 `(chunk_size, max_action_dim)` 的 action。

In [11]:
from torch.utils.data.dataloader import default_collate

batch = default_collate([sample])

def to_device(batch, device):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

batch = to_device(batch, device)
print("batch action shape:", batch["action"].shape)  # [B, chunk_size, max_action_dim]
print("batch observation.images.image0 shape:", batch["observation.images.image0"].shape)  # [B, T, C, H, W]

batch action shape: torch.Size([1, 50, 32])
batch observation.images.image0 shape: torch.Size([1, 3, 3, 224, 224])


In [17]:
batch['observation.pixel_values'].shape

torch.Size([1, 768, 1536])

## 3. 推理

`select_action(batch)` 是推理入口：内部调用 `predict_action_chunk` 做 flow-matching 去噪，再按 `n_action_steps` 逐步弹出 action。

> TBot 默认 `dtype=bfloat16`，notebook 推理建议包一层 `torch.autocast`，与训练 AMP 行为一致。

In [12]:
# 推理 batch 可以去掉 action（select_action 不读取 action）
infer_batch = {k: v for k, v in batch.items() if k != "action"}

with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    pred_action = policy.select_action(infer_batch)

# pred_action: [batch_size, action_dim]，已截断到真实维度（aloha=14，其余为 padding）
print("pred_action shape:", pred_action.shape)
print("pred_action:", pred_action)

pred_action shape: torch.Size([1, 32])
pred_action: tensor([[ 5.2398e-02, -1.0201e-01, -7.6659e-02, -3.9336e-02,  8.5267e-02,
          3.2438e-02,  8.0241e-01, -1.6632e-01, -2.2536e-01, -1.9258e-01,
          4.6247e-02,  2.5375e-02, -5.0865e-02,  7.8227e-01, -3.6629e-03,
         -1.0296e-03, -2.8908e-04, -1.9188e-03, -4.6921e-03, -3.6969e-03,
         -3.1722e-04, -1.7295e-03,  2.5551e-03, -4.9517e-04,  4.6349e-04,
          3.1878e-03, -2.4685e-03,  5.8153e-04, -8.5294e-05,  3.4715e-03,
         -7.2810e-04,  6.4054e-03]], device='cuda:0')


In [13]:
# 一次预测完整 action chunk（chunk_size 步）
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    action_chunk, recon_images = policy.predict_action_chunk(infer_batch)

original_action_dim = policy.config.output_features["action"].shape[0]
action_chunk = action_chunk[:, :, :original_action_dim]
print("action_chunk shape:", action_chunk.shape)  # [B, n_action_steps, action_dim]
print("first step action:", action_chunk[0, 0])

action_chunk shape: torch.Size([1, 50, 32])
first step action: tensor([ 7.0393e-02, -1.3606e-01, -1.0507e-01, -2.4307e-03, -2.3220e-02,
         6.3752e-02,  8.0476e-01, -1.5775e-01, -2.2555e-01, -1.9275e-01,
         4.3530e-02,  2.3688e-02, -5.3436e-02,  7.7410e-01, -4.2688e-03,
        -3.4024e-03,  4.7074e-04, -2.6665e-03,  8.4275e-04,  1.2216e-04,
         2.4555e-03,  6.2242e-04,  4.1513e-03, -5.5261e-05,  6.5517e-04,
        -4.5484e-04, -1.3019e-03,  8.6540e-04,  2.3508e-03,  2.9708e-03,
         2.5465e-03,  1.2426e-03], device='cuda:0')


## 4. 训练 forward

`forward(batch)` 需要 batch 中包含 **完整 action chunk**（shape `[B, chunk_size, max_action_dim]`），由 `delta_timestamps` + transform 链保证。

返回 `(loss, loss_dict)`，其中总 loss = `loss_action + λ_gen·loss_gen + λ_3d·loss_3d`。

In [14]:
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    loss, output_dict = policy.forward(batch)

print("loss:", loss.item())
print("output keys:", sorted(output_dict.keys()))
print("loss_action:", output_dict.get("loss_action"))
print("loss_gen:", output_dict.get("loss_gen"))
print("loss_3d:", output_dict.get("loss_3d"))

[INFO ] Selecting reference view using strategy: saddle_balanced
loss: 0.19527968764305115
output keys: ['loss', 'loss_3d', 'loss_3d_q13_t11', 'loss_3d_q19_t15', 'loss_3d_q23_t19', 'loss_3d_q27_t23', 'loss_action', 'loss_action_dim0', 'loss_action_dim1', 'loss_action_dim10', 'loss_action_dim11', 'loss_action_dim12', 'loss_action_dim13', 'loss_action_dim14', 'loss_action_dim15', 'loss_action_dim16', 'loss_action_dim17', 'loss_action_dim18', 'loss_action_dim19', 'loss_action_dim2', 'loss_action_dim20', 'loss_action_dim21', 'loss_action_dim22', 'loss_action_dim23', 'loss_action_dim24', 'loss_action_dim25', 'loss_action_dim26', 'loss_action_dim27', 'loss_action_dim28', 'loss_action_dim29', 'loss_action_dim3', 'loss_action_dim30', 'loss_action_dim31', 'loss_action_dim4', 'loss_action_dim5', 'loss_action_dim6', 'loss_action_dim7', 'loss_action_dim8', 'loss_action_dim9', 'loss_gen', 'time_3d_teacher_forward_s']
loss_action: 0.18599994480609894
loss_gen: 0.5947919487953186
loss_3d: 0.333182990

# 训练拆解

In [18]:
batch['observation.pixel_values'].shape

torch.Size([1, 768, 1536])

## 4.1 展开版 forward 调试

下面把 `TBotSA1Policy.forward()` 和 `TBotSA1Model.forward()` 拆成多个可单独运行的 cell。所有中间变量都保留在 notebook 全局命名空间，方便在任意阶段插入额外打印。

In [19]:
import torch
import torch.nn.functional as F

from lerobot.utils.constants import ACTION, OBS_PREFIX, SAMPLE_ACTION_LOSS_MASK


def show(name, x):
    if isinstance(x, torch.Tensor):
        print(f"{name:<42} shape={tuple(x.shape)}, dtype={x.dtype}, device={x.device}")
    else:
        print(f"{name:<42} {type(x)}")


# 使用包含 action 的训练 batch；不要传 infer_batch。
debug_batch = batch

device = next(policy.parameters()).device
debug_batch = {
    k: (v.to(device) if isinstance(v, torch.Tensor) else v)
    for k, v in debug_batch.items()
}

model = policy.model
policy.train(False)
print("ready")

ready


输入的batch准备

In [20]:
print("========== 1. Policy.forward: parse batch ==========")

pixel_values = debug_batch[f"{OBS_PREFIX}pixel_values"]
image_grid_thw = debug_batch[f"{OBS_PREFIX}image_grid_thw"]
lang_tokens = debug_batch[f"{OBS_PREFIX}input_ids"]
lang_masks = debug_batch[f"{OBS_PREFIX}attention_mask"]

images, img_masks = policy._preprocess_images(debug_batch)
state = policy.prepare_state(debug_batch)
actions = policy.prepare_action(debug_batch)

show("pixel_values", pixel_values)
show("image_grid_thw", image_grid_thw)
show("lang_tokens/input_ids", lang_tokens)
show("lang_masks/attention_mask", lang_masks)
show("images", images)
show("img_masks", img_masks)
show("state", state)
show("actions", actions)

========== 1. Policy.forward: parse batch ==========
pixel_values                               shape=(1, 768, 1536), dtype=torch.float32, device=cuda:0
image_grid_thw                             shape=(1, 3, 3), dtype=torch.int64, device=cuda:0
lang_tokens/input_ids                      shape=(1, 246), dtype=torch.int64, device=cuda:0
lang_masks/attention_mask                  shape=(1, 246), dtype=torch.int64, device=cuda:0
images                                     shape=(1, 3, 3, 3, 224, 224), dtype=torch.float32, device=cuda:0
img_masks                                  shape=(1, 3), dtype=torch.bool, device=cuda:0
state                                      shape=(1, 32), dtype=torch.float32, device=cuda:0
actions                                    shape=(1, 50, 32), dtype=torch.float32, device=cuda:0


In [21]:
print("========== 2. TBotSA1Model.forward: flow matching ==========")

noise = model.sample_noise(actions.shape, actions.device)
time = model.sample_time(actions.shape[0], actions.device)

time_expanded = time[:, None, None]
x_t = time_expanded * noise + (1 - time_expanded) * actions
u_t = noise - actions

show("noise", noise)
show("time", time)
show("x_t noisy actions", x_t)
show("u_t target velocity", u_t)

========== 2. TBotSA1Model.forward: flow matching ==========
noise                                      shape=(1, 50, 32), dtype=torch.float32, device=cuda:0
time                                       shape=(1,), dtype=torch.float32, device=cuda:0
x_t noisy actions                          shape=(1, 50, 32), dtype=torch.float32, device=cuda:0
u_t target velocity                        shape=(1, 50, 32), dtype=torch.float32, device=cuda:0


In [22]:
print("========== 3. embed_prefix ==========")

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    prefix_embs, prefix_pad_masks, prefix_att_masks = model.embed_prefix(
        pixel_values,
        image_grid_thw,
        lang_tokens,
        lang_masks,
    )

show("prefix_embs", prefix_embs)
show("prefix_pad_masks", prefix_pad_masks)
show("prefix_att_masks", prefix_att_masks)

========== 3. embed_prefix ==========
prefix_embs                                shape=(1, 246, 2048), dtype=torch.bfloat16, device=cuda:0
prefix_pad_masks                           shape=(1, 246), dtype=torch.bool, device=cuda:0
prefix_att_masks                           shape=(1, 246), dtype=torch.bool, device=cuda:0


In [23]:
print("========== 4. embed_middle ==========")

middle_input_images = images[:, :, :2]
future_images = images[:, :, 2]

show("middle_input_images images[:, :, :2]", middle_input_images)
show("future_images images[:, :, 2]", future_images)

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    middle_embs, middle_pad_masks, middle_att_masks = model.embed_middle(
        middle_input_images,
        img_masks,
    )

show("middle_embs", middle_embs)
show("middle_pad_masks", middle_pad_masks)
show("middle_att_masks", middle_att_masks)
print(f"middle_visual_token_count = {model.middle_visual_token_count}")
print(f"middle_query_token_count  = {model.middle_query_token_count}")
print(f"middle_total_token_count  = {model.middle_total_token_count}")

========== 4. embed_middle ==========
middle_input_images images[:, :, :2]       shape=(1, 3, 2, 3, 224, 224), dtype=torch.float32, device=cuda:0
future_images images[:, :, 2]              shape=(1, 3, 3, 224, 224), dtype=torch.float32, device=cuda:0
middle_embs                                shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
middle_pad_masks                           shape=(1, 528), dtype=torch.bool, device=cuda:0
middle_att_masks                           shape=(1, 528), dtype=torch.bool, device=cuda:0
middle_visual_token_count = 96
middle_query_token_count  = 432
middle_total_token_count  = 528


In [25]:
print("========== 5. embed_suffix ==========")

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    suffix_embs, suffix_pad_masks, suffix_att_masks = model.embed_suffix(
        state,
        x_t,
        time,
    )

show("suffix_embs", suffix_embs)
show("suffix_pad_masks", suffix_pad_masks)
show("suffix_att_masks", suffix_att_masks)

========== 5. embed_suffix ==========
suffix_embs                                shape=(1, 51, 1024), dtype=torch.bfloat16, device=cuda:0
suffix_pad_masks                           shape=(1, 51), dtype=torch.bool, device=cuda:0
suffix_att_masks                           shape=(1, 51), dtype=torch.bfloat16, device=cuda:0


In [26]:
print("========== 6. dtype align + concat masks ==========")

if (
    model.qwen3_vl_with_expert.und_expert.language_model.layers[0].self_attn.q_proj.weight.dtype
    == torch.bfloat16
):
    suffix_embs = suffix_embs.to(dtype=torch.bfloat16)
    middle_embs = middle_embs.to(dtype=torch.bfloat16)
    prefix_embs = prefix_embs.to(dtype=torch.bfloat16)

pad_masks = torch.cat([prefix_pad_masks, middle_pad_masks, suffix_pad_masks], dim=1)
att_masks = torch.cat([prefix_att_masks, middle_att_masks, suffix_att_masks], dim=1)

show("prefix_embs after dtype align", prefix_embs)
show("middle_embs after dtype align", middle_embs)
show("suffix_embs after dtype align", suffix_embs)
show("pad_masks full", pad_masks)
show("att_masks full", att_masks)

print(f"prefix_len = {prefix_pad_masks.shape[1]}")
print(f"middle_len = {middle_pad_masks.shape[1]}")
print(f"suffix_len = {suffix_pad_masks.shape[1]}")
print(f"total_len  = {pad_masks.shape[1]}")

========== 6. dtype align + concat masks ==========
prefix_embs after dtype align              shape=(1, 246, 2048), dtype=torch.bfloat16, device=cuda:0
middle_embs after dtype align              shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
suffix_embs after dtype align              shape=(1, 51, 1024), dtype=torch.bfloat16, device=cuda:0
pad_masks full                             shape=(1, 825), dtype=torch.bool, device=cuda:0
att_masks full                             shape=(1, 825), dtype=torch.bfloat16, device=cuda:0
prefix_len = 246
middle_len = 528
suffix_len = 51
total_len  = 825


In [27]:
print("========== 7. attention mask + position ids ==========")

att_2d_masks = model.build_training_attention_mask(
    pad_masks,
    att_masks,
    prefix_len=prefix_pad_masks.shape[1],
)
position_ids, rope_deltas = model.get_position_ids(
    lang_tokens,
    image_grid_thw,
    pad_masks,
)
att_2d_masks_4d = model._prepare_attention_masks_4d(att_2d_masks)

show("att_2d_masks", att_2d_masks)
show("position_ids", position_ids)
show("rope_deltas", rope_deltas)
show("att_2d_masks_4d", att_2d_masks_4d)

========== 7. attention mask + position ids ==========
att_2d_masks                               shape=(1, 825, 825), dtype=torch.bool, device=cuda:0
position_ids                               shape=(3, 1, 825), dtype=torch.int64, device=cuda:0
rope_deltas                                shape=(1, 1), dtype=torch.int64, device=cuda:0
att_2d_masks_4d                            shape=(1, 1, 825, 825), dtype=torch.float32, device=cuda:0


In [28]:
print("========== 8. MoT forward ==========")

collect_middle_layers = model.query_layer_indices if model.da3_teacher is not None else None
print("collect_middle_layers =", collect_middle_layers)

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    outputs = model.qwen3_vl_with_expert.forward(
        attention_mask=att_2d_masks_4d,
        position_ids=position_ids,
        past_key_values=None,
        inputs_embeds=[prefix_embs, middle_embs, suffix_embs],
        use_cache=False,
        collect_middle_layers=collect_middle_layers,
    )

if collect_middle_layers is None:
    (_, middle_out, suffix_out), _ = outputs
    middle_layer_outputs = ()
else:
    (_, middle_out, suffix_out), _, middle_layer_outputs = outputs

show("middle_out", middle_out)
show("suffix_out full", suffix_out)
for i, middle_layer_output in enumerate(middle_layer_outputs):
    show(f"middle_layer_outputs[{i}]", middle_layer_output)

========== 8. MoT forward ==========
collect_middle_layers = (13, 19, 23, 27)
middle_out                                 shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
suffix_out full                            shape=(1, 51, 1024), dtype=torch.bfloat16, device=cuda:0
middle_layer_outputs[0]                    shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
middle_layer_outputs[1]                    shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
middle_layer_outputs[2]                    shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
middle_layer_outputs[3]                    shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0


In [29]:
print("========== 9. loss_gen ==========")

if float(policy.config.lambda_gen) > 0.0:
    middle_visual_out, middle_query_out = model.split_middle_tokens(middle_out)
    show("middle_visual_out", middle_visual_out)
    show("middle_query_out", middle_query_out)

    with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
        pred_cosmos_features = model.decode_cosmos(middle_visual_out.to(dtype=torch.float32))
        future_embs = model.get_cosmos_features(future_images)

    show("pred_cosmos_features", pred_cosmos_features)
    show("future_embs", future_embs)

    loss_gen = F.mse_loss(
        pred_cosmos_features[img_masks],
        future_embs.to(dtype=torch.float32)[img_masks],
    )
else:
    middle_visual_out, middle_query_out = None, None
    loss_gen = middle_out.new_zeros((), dtype=torch.float32)

print("loss_gen =", loss_gen.item())

========== 9. loss_gen ==========
middle_visual_out                          shape=(1, 96, 1024), dtype=torch.bfloat16, device=cuda:0
middle_query_out                           shape=(1, 432, 1024), dtype=torch.bfloat16, device=cuda:0
pred_cosmos_features                       shape=(1, 3, 16, 32, 32), dtype=torch.bfloat16, device=cuda:0
future_embs                                shape=(1, 3, 16, 32, 32), dtype=torch.float32, device=cuda:0
loss_gen = 0.5948032736778259


In [30]:
print("========== 10. loss_3d ==========")

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    loss_3d, loss_3d_logs = model.compute_3d_query_loss(
        tuple(middle_layer_outputs),
        future_images,
        img_masks,
    )

print("loss_3d =", loss_3d.item())
for key, value in loss_3d_logs.items():
    print(key, "=", value.item())

========== 10. loss_3d ==========
loss_3d = 0.3331829905509949
time_3d_teacher_forward_s = 0.048203326761722565
loss_3d_q13_t11 = 0.08984170854091644
loss_3d_q19_t15 = 0.20179657638072968
loss_3d_q23_t19 = 0.40640878677368164
loss_3d_q27_t23 = 0.6346849799156189


In [31]:
print("========== 11. loss_action ==========")

suffix_out_action = suffix_out[:, -policy.config.chunk_size :]
suffix_out_action = suffix_out_action.to(dtype=torch.float32)
show("suffix_out_action", suffix_out_action)

v_t = model.action_out_proj(suffix_out_action)
show("v_t predicted velocity", v_t)

losses_action = F.mse_loss(u_t, v_t, reduction="none")
show("losses_action raw", losses_action)

========== 11. loss_action ==========
suffix_out_action                          shape=(1, 50, 1024), dtype=torch.float32, device=cuda:0
v_t predicted velocity                     shape=(1, 50, 32), dtype=torch.float32, device=cuda:0
losses_action raw                          shape=(1, 50, 32), dtype=torch.float32, device=cuda:0


In [32]:
print("========== 12. Policy.forward: aggregate final loss ==========")

original_action_dim = policy.config.output_features[ACTION].shape[0]
losses_action = losses_action[:, :, :original_action_dim]
show("losses_action clipped", losses_action)
print("original_action_dim =", original_action_dim)

action_loss_mask = debug_batch.get(SAMPLE_ACTION_LOSS_MASK)
if action_loss_mask is None:
    action_loss_mask = torch.ones(
        losses_action.shape[0],
        dtype=torch.bool,
        device=losses_action.device,
    )
else:
    action_loss_mask = action_loss_mask.to(losses_action.device)
    if action_loss_mask.ndim > 1:
        action_loss_mask = action_loss_mask.squeeze(-1)
    action_loss_mask = action_loss_mask > 0.5

show("action_loss_mask", action_loss_mask)

if action_loss_mask.any():
    sample_losses_action = losses_action[action_loss_mask]
    show("sample_losses_action", sample_losses_action)

    if policy.config.mask_action_dim_padding_loss:
        valid_action_dim = min(int(policy.config.action_loss_valid_dim), original_action_dim)
        valid_losses_action = sample_losses_action[:, :, :valid_action_dim]
        loss_action = valid_losses_action.mean()
        loss_action_by_dim_tensor = sample_losses_action.new_zeros(original_action_dim)
        loss_action_by_dim_tensor[:valid_action_dim] = valid_losses_action.mean(dim=[0, 1])
        loss_action_by_dim = loss_action_by_dim_tensor.detach().cpu().numpy().tolist()
    else:
        loss_action = sample_losses_action.mean()
        loss_action_by_dim = sample_losses_action.mean(dim=[0, 1]).detach().cpu().numpy().tolist()
else:
    loss_action = losses_action.new_zeros(())
    loss_action_by_dim = [0.0] * original_action_dim

loss = loss_action + policy.config.lambda_gen * loss_gen + policy.config.lambda_3d * loss_3d

loss_dict_debug = {
    "loss": loss.item(),
    "loss_action": loss_action.item(),
    "loss_gen": loss_gen.item(),
    "loss_3d": loss_3d.item(),
}
for key, value in loss_3d_logs.items():
    loss_dict_debug[key] = float(value.item())

loss_dict_debug.update({
    f"loss_action_dim{i}": loss_action_by_dim[i]
    for i in range(original_action_dim)
})

print("loss =", loss.item())
print("loss_action =", loss_action.item())
print("loss_gen =", loss_gen.item())
print("loss_3d =", loss_3d.item())
loss_dict_debug

========== 12. Policy.forward: aggregate final loss ==========
losses_action clipped                      shape=(1, 50, 32), dtype=torch.float32, device=cuda:0
original_action_dim = 32
action_loss_mask                           shape=(1,), dtype=torch.bool, device=cuda:0
sample_losses_action                       shape=(1, 50, 32), dtype=torch.float32, device=cuda:0
loss = 0.15280824899673462
loss_action = 0.14352838695049286
loss_gen = 0.5948032736778259
loss_3d = 0.3331829905509949


{'loss': 0.15280824899673462,
 'loss_action': 0.14352838695049286,
 'loss_gen': 0.5948032736778259,
 'loss_3d': 0.3331829905509949,
 'time_3d_teacher_forward_s': 0.048203326761722565,
 'loss_3d_q13_t11': 0.08984170854091644,
 'loss_3d_q19_t15': 0.20179657638072968,
 'loss_3d_q23_t19': 0.40640878677368164,
 'loss_3d_q27_t23': 0.6346849799156189,
 'loss_action_dim0': 0.7840615510940552,
 'loss_action_dim1': 0.7025982737541199,
 'loss_action_dim2': 0.48856300115585327,
 'loss_action_dim3': 0.3399224877357483,
 'loss_action_dim4': 0.22807537019252777,
 'loss_action_dim5': 0.31225401163101196,
 'loss_action_dim6': 0.025456862524151802,
 'loss_action_dim7': 0.21909894049167633,
 'loss_action_dim8': 0.056783318519592285,
 'loss_action_dim9': 0.06313307583332062,
 'loss_action_dim10': 0.039663165807724,
 'loss_action_dim11': 0.5041913986206055,
 'loss_action_dim12': 0.5214625597000122,
 'loss_action_dim13': 0.29715627431869507,
 'loss_action_dim14': 0.0004961880040355027,
 'loss_action_dim15':

说明：上面的展开版 forward 会重新采样 `noise` 和 `time`，因此数值不一定和前面直接 `policy.forward(batch)` 的 loss 完全一致。若要严格对齐，可以固定随机种子后只运行这一路，或临时调用 `model.forward(..., noise=noise, time=time)` 做对照。

## Behavior Prompt改造验证 方案1（废弃）

这版只把 state/action learned tokens 拼到 prefix，没有扩增 `pixel_values`，且 action token 数与 behavior 图像块数没有对齐。后续请跳过这一节，使用下面的“方案1 修正版”。

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from lerobot.utils.constants import ACTION, OBS_PREFIX, SAMPLE_ACTION_LOSS_MASK


def show_bp(name, x):
    if isinstance(x, torch.Tensor):
        print(f"{name:<46} shape={tuple(x.shape)}, dtype={x.dtype}, device={x.device}")
    else:
        print(f"{name:<46} {type(x)}")


# 方案1只验证 state/action learned prompt token 拼到 prefix_embs 后是否能完整前向。
# 图像 behavior prompt 暂不放进 pixel_values，避免同时处理 Qwen image placeholder 对齐问题。
bp_batch = {
    k: (v.to(device) if isinstance(v, torch.Tensor) else v)
    for k, v in batch.items()
}
bp_model = policy.model
bp_hidden_dim = bp_model.qwen3_vl_with_expert.und_expert.language_model.config.hidden_size
bp_prompt_tokens = 8

print("bp_hidden_dim =", bp_hidden_dim)
print("bp_prompt_tokens =", bp_prompt_tokens)

bp_hidden_dim = 2048
bp_prompt_tokens = 8


In [34]:
print("========== BP 方案1 / 1. 构造 state-action prompt tokens ==========")

bp_pixel_values = bp_batch[f"{OBS_PREFIX}pixel_values"]
bp_image_grid_thw = bp_batch[f"{OBS_PREFIX}image_grid_thw"]
bp_lang_tokens = bp_batch[f"{OBS_PREFIX}input_ids"]
bp_lang_masks = bp_batch[f"{OBS_PREFIX}attention_mask"]

bp_images, bp_img_masks = policy._preprocess_images(bp_batch)
bp_state = policy.prepare_state(bp_batch)
bp_actions = policy.prepare_action(bp_batch)

# 用当前样本伪造 behavior prompt：state + action chunk。
# 这里先完整保留 action 序列，不做时间下采样；MLP 会把 50 个 action token 投到 prefix hidden dim。
bp_state_prompt = bp_state[:, None, :]
bp_action_prompt = bp_actions

bp_state_proj = nn.Linear(bp_state_prompt.shape[-1], bp_hidden_dim, device=device, dtype=torch.float32)
bp_action_proj = nn.Linear(bp_action_prompt.shape[-1], bp_hidden_dim, device=device, dtype=torch.float32)
bp_prompt_pool = nn.AdaptiveAvgPool1d(bp_prompt_tokens)

bp_state_prompt_embs = bp_state_proj(bp_state_prompt.to(torch.float32))
bp_action_prompt_embs_full = bp_action_proj(bp_action_prompt.to(torch.float32))
# (B, T, D) -> (B, K, D)，先把完整 action 压缩成固定数量 prefix prompt tokens。
bp_action_prompt_embs = bp_prompt_pool(
    bp_action_prompt_embs_full.transpose(1, 2)
).transpose(1, 2)

bp_prompt_embs = torch.cat([bp_state_prompt_embs, bp_action_prompt_embs], dim=1)
bp_prompt_mask = torch.ones(
    bp_prompt_embs.shape[:2],
    dtype=torch.bool,
    device=bp_prompt_embs.device,
)
bp_prompt_att_mask = torch.zeros_like(bp_prompt_mask)

show_bp("bp_pixel_values", bp_pixel_values)
show_bp("bp_image_grid_thw", bp_image_grid_thw)
show_bp("bp_lang_tokens", bp_lang_tokens)
show_bp("bp_lang_masks", bp_lang_masks)
show_bp("bp_state", bp_state)
show_bp("bp_actions", bp_actions)
show_bp("bp_state_prompt_embs", bp_state_prompt_embs)
show_bp("bp_action_prompt_embs_full", bp_action_prompt_embs_full)
show_bp("bp_action_prompt_embs", bp_action_prompt_embs)
show_bp("bp_prompt_embs", bp_prompt_embs)
show_bp("bp_prompt_mask", bp_prompt_mask)

========== BP 方案1 / 1. 构造 state-action prompt tokens ==========
bp_pixel_values                                shape=(1, 768, 1536), dtype=torch.float32, device=cuda:0
bp_image_grid_thw                              shape=(1, 3, 3), dtype=torch.int64, device=cuda:0
bp_lang_tokens                                 shape=(1, 246), dtype=torch.int64, device=cuda:0
bp_lang_masks                                  shape=(1, 246), dtype=torch.int64, device=cuda:0
bp_state                                       shape=(1, 32), dtype=torch.float32, device=cuda:0
bp_actions                                     shape=(1, 50, 32), dtype=torch.float32, device=cuda:0
bp_state_prompt_embs                           shape=(1, 1, 2048), dtype=torch.float32, device=cuda:0
bp_action_prompt_embs_full                     shape=(1, 50, 2048), dtype=torch.float32, device=cuda:0
bp_action_prompt_embs                          shape=(1, 8, 2048), dtype=torch.float32, device=cuda:0
bp_prompt_embs                        

In [35]:
print("========== BP 方案1 / 2. embed_prefix 后拼接 prompt tokens ==========")

bp_noise = bp_model.sample_noise(bp_actions.shape, bp_actions.device)
bp_time = bp_model.sample_time(bp_actions.shape[0], bp_actions.device)
bp_x_t = bp_time[:, None, None] * bp_noise + (1 - bp_time[:, None, None]) * bp_actions
bp_u_t = bp_noise - bp_actions

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    bp_prefix_embs, bp_prefix_pad_masks, bp_prefix_att_masks = bp_model.embed_prefix(
        bp_pixel_values,
        bp_image_grid_thw,
        bp_lang_tokens,
        bp_lang_masks,
    )
    bp_middle_embs, bp_middle_pad_masks, bp_middle_att_masks = bp_model.embed_middle(
        bp_images[:, :, :2],
        bp_img_masks,
    )
    bp_suffix_embs, bp_suffix_pad_masks, bp_suffix_att_masks = bp_model.embed_suffix(
        bp_state,
        bp_x_t,
        bp_time,
    )

# learned prompt tokens 拼在 prefix 最前面，作为 behavior prompt 条件。
bp_prompt_embs = bp_prompt_embs.to(dtype=bp_prefix_embs.dtype)
bp_prefix_embs_ext = torch.cat([bp_prompt_embs, bp_prefix_embs], dim=1)
bp_prefix_pad_masks_ext = torch.cat([bp_prompt_mask, bp_prefix_pad_masks], dim=1)
bp_prefix_att_masks_ext = torch.cat([bp_prompt_att_mask, bp_prefix_att_masks], dim=1)

# get_position_ids 需要 lang_tokens 长度覆盖 prefix tokens；用 pseudo token id 为 learned prompt 占位。
bp_pseudo_token_id = 777
bp_prompt_token_ids = torch.full(
    bp_prompt_mask.shape,
    bp_pseudo_token_id,
    dtype=bp_lang_tokens.dtype,
    device=bp_lang_tokens.device,
)
bp_lang_tokens_ext = torch.cat([bp_prompt_token_ids, bp_lang_tokens], dim=1)
bp_lang_masks_ext = torch.cat([bp_prompt_mask.to(bp_lang_masks.dtype), bp_lang_masks], dim=1)

if (
    bp_model.qwen3_vl_with_expert.und_expert.language_model.layers[0].self_attn.q_proj.weight.dtype
    == torch.bfloat16
):
    bp_prefix_embs_ext = bp_prefix_embs_ext.to(dtype=torch.bfloat16)
    bp_middle_embs = bp_middle_embs.to(dtype=torch.bfloat16)
    bp_suffix_embs = bp_suffix_embs.to(dtype=torch.bfloat16)

bp_pad_masks = torch.cat([bp_prefix_pad_masks_ext, bp_middle_pad_masks, bp_suffix_pad_masks], dim=1)
bp_att_masks = torch.cat([bp_prefix_att_masks_ext, bp_middle_att_masks, bp_suffix_att_masks], dim=1)

bp_att_2d_masks = bp_model.build_training_attention_mask(
    bp_pad_masks,
    bp_att_masks,
    prefix_len=bp_prefix_pad_masks_ext.shape[1],
)
bp_position_ids, bp_rope_deltas = bp_model.get_position_ids(
    bp_lang_tokens_ext,
    bp_image_grid_thw,
    bp_pad_masks,
)
bp_att_2d_masks_4d = bp_model._prepare_attention_masks_4d(bp_att_2d_masks)

show_bp("bp_prefix_embs", bp_prefix_embs)
show_bp("bp_prefix_embs_ext", bp_prefix_embs_ext)
show_bp("bp_lang_tokens_ext", bp_lang_tokens_ext)
show_bp("bp_lang_masks_ext", bp_lang_masks_ext)
show_bp("bp_middle_embs", bp_middle_embs)
show_bp("bp_suffix_embs", bp_suffix_embs)
show_bp("bp_pad_masks", bp_pad_masks)
show_bp("bp_att_2d_masks", bp_att_2d_masks)
show_bp("bp_position_ids", bp_position_ids)
show_bp("bp_att_2d_masks_4d", bp_att_2d_masks_4d)
print("bp prefix original/ext:", bp_prefix_embs.shape[1], "->", bp_prefix_embs_ext.shape[1])
print("bp total_len:", bp_pad_masks.shape[1])

========== BP 方案1 / 2. embed_prefix 后拼接 prompt tokens ==========
bp_prefix_embs                                 shape=(1, 246, 2048), dtype=torch.bfloat16, device=cuda:0
bp_prefix_embs_ext                             shape=(1, 255, 2048), dtype=torch.bfloat16, device=cuda:0
bp_lang_tokens_ext                             shape=(1, 255), dtype=torch.int64, device=cuda:0
bp_lang_masks_ext                              shape=(1, 255), dtype=torch.int64, device=cuda:0
bp_middle_embs                                 shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp_suffix_embs                                 shape=(1, 51, 1024), dtype=torch.bfloat16, device=cuda:0
bp_pad_masks                                   shape=(1, 834), dtype=torch.bool, device=cuda:0
bp_att_2d_masks                                shape=(1, 834, 834), dtype=torch.bool, device=cuda:0
bp_position_ids                                shape=(3, 1, 834), dtype=torch.int64, device=cuda:0
bp_att_2d_masks_4d            

In [36]:
print("========== BP 方案1 / 3. MoT forward + loss ==========")

bp_collect_middle_layers = bp_model.query_layer_indices if bp_model.da3_teacher is not None else None
print("bp_collect_middle_layers =", bp_collect_middle_layers)

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    bp_outputs = bp_model.qwen3_vl_with_expert.forward(
        attention_mask=bp_att_2d_masks_4d,
        position_ids=bp_position_ids,
        past_key_values=None,
        inputs_embeds=[bp_prefix_embs_ext, bp_middle_embs, bp_suffix_embs],
        use_cache=False,
        collect_middle_layers=bp_collect_middle_layers,
    )

if bp_collect_middle_layers is None:
    (_, bp_middle_out, bp_suffix_out), _ = bp_outputs
    bp_middle_layer_outputs = ()
else:
    (_, bp_middle_out, bp_suffix_out), _, bp_middle_layer_outputs = bp_outputs

show_bp("bp_middle_out", bp_middle_out)
show_bp("bp_suffix_out", bp_suffix_out)
for i, x in enumerate(bp_middle_layer_outputs):
    show_bp(f"bp_middle_layer_outputs[{i}]", x)

if float(policy.config.lambda_gen) > 0.0:
    bp_middle_visual_out, bp_middle_query_out = bp_model.split_middle_tokens(bp_middle_out)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
        bp_pred_cosmos_features = bp_model.decode_cosmos(bp_middle_visual_out.to(dtype=torch.float32))
        bp_future_embs = bp_model.get_cosmos_features(bp_images[:, :, 2])
    bp_loss_gen = F.mse_loss(
        bp_pred_cosmos_features[bp_img_masks],
        bp_future_embs.to(dtype=torch.float32)[bp_img_masks],
    )
else:
    bp_loss_gen = bp_middle_out.new_zeros((), dtype=torch.float32)

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    bp_loss_3d, bp_loss_3d_logs = bp_model.compute_3d_query_loss(
        tuple(bp_middle_layer_outputs),
        bp_images[:, :, 2],
        bp_img_masks,
    )

bp_suffix_out_action = bp_suffix_out[:, -policy.config.chunk_size :].to(dtype=torch.float32)
bp_v_t = bp_model.action_out_proj(bp_suffix_out_action)
bp_losses_action = F.mse_loss(bp_u_t, bp_v_t, reduction="none")

show_bp("bp_middle_visual_out", bp_middle_visual_out if float(policy.config.lambda_gen) > 0.0 else None)
show_bp("bp_suffix_out_action", bp_suffix_out_action)
show_bp("bp_v_t", bp_v_t)
show_bp("bp_losses_action raw", bp_losses_action)
print("bp_loss_gen =", bp_loss_gen.item())
print("bp_loss_3d =", bp_loss_3d.item())

========== BP 方案1 / 3. MoT forward + loss ==========
bp_collect_middle_layers = (13, 19, 23, 27)
bp_middle_out                                  shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp_suffix_out                                  shape=(1, 51, 1024), dtype=torch.bfloat16, device=cuda:0
bp_middle_layer_outputs[0]                     shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp_middle_layer_outputs[1]                     shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp_middle_layer_outputs[2]                     shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp_middle_layer_outputs[3]                     shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp_middle_visual_out                           shape=(1, 96, 1024), dtype=torch.bfloat16, device=cuda:0
bp_suffix_out_action                           shape=(1, 50, 1024), dtype=torch.float32, device=cuda:0
bp_v_t                                         shape=(1, 50, 32), d

方案1说明：这个验证只把 state/action 伪造的 learned prompt tokens 拼到 `prefix_embs`，不改 `pixel_values`。因此它验证的是“behavior prompt 作为 prefix learned tokens 后，attention mask、position ids、MoT forward 和 loss 是否能跑通”。后续如果要把轨迹图像也作为 behavior prompt，需要额外同步扩展 `pixel_values`、`image_grid_thw` 和 `lang_tokens` 里的 image placeholder。

## Behavior Prompt改造验证 方案1 修正版

这个方案把当前三路图像复制 4 次作为 behavior prompt 图像，再保留原始三路图像作为当前观测。`lang_tokens` 不再复用原始 task 文本，而是重建为纯 image placeholder，因此最终 `prefix_embs` 不包含 task token 信息。

state/action prompt 与 4 个 behavior 图像块对齐：把 50 步 action chunk 切成 4 段，每段聚合后与同一份当前 state 拼接，再通过 MLP 得到 `action_state_emb: (B, 4, 2048)`。这 4 个 token 分别对应 4 个 behavior prompt 图像块。

In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from lerobot.utils.constants import ACTION, OBS_PREFIX, SAMPLE_ACTION_LOSS_MASK

def show_bp2(name, x, note=""):
    if isinstance(x, torch.Tensor):
        suffix = f"  # {note}" if note else ""
        print(f"{name:<50} shape={tuple(x.shape)}, dtype={x.dtype}, device={x.device}{suffix}")
    else:
        print(f"{name:<50} {type(x)}")

bp2_batch = {
    k: (v.to(device) if isinstance(v, torch.Tensor) else v)
    for k, v in batch.items()
}
bp2_model = policy.model
bp2_hidden_dim = bp2_model.qwen3_vl_with_expert.und_expert.language_model.config.hidden_size

# 4 个 behavior 图像块 + 1 个当前观测图像块。
bp2_behavior_chunks = 4
bp2_behavior_image_repeats = bp2_behavior_chunks
bp2_total_image_groups = bp2_behavior_chunks + 1

print("bp2_hidden_dim =", bp2_hidden_dim)
print("bp2_behavior_chunks =", bp2_behavior_chunks)
print("bp2_total_image_groups =", bp2_total_image_groups)

bp2_hidden_dim = 2048
bp2_behavior_chunks = 4
bp2_total_image_groups = 5


In [44]:
print("========== BP 方案1修正版 / 1. 扩增 pixel_values 并重建无 task 的 image-only lang_tokens ==========")

bp2_pixel_values = bp2_batch[f"{OBS_PREFIX}pixel_values"]
bp2_image_grid_thw = bp2_batch[f"{OBS_PREFIX}image_grid_thw"]
bp2_state = policy.prepare_state(bp2_batch)
bp2_actions = policy.prepare_action(bp2_batch)
bp2_images, bp2_img_masks = policy._preprocess_images(bp2_batch)

# 当前三路图像复制 4 次作为 behavior prompt，再加原始当前三路图像。
# pixel_values: (B, P, 1536) -> (B, P * 5, 1536)
# image_grid_thw: (B, 3, 3) -> (B, 15, 3)
bp2_pixel_values_ext = bp2_pixel_values.repeat(1, bp2_total_image_groups, 1)
bp2_image_grid_thw_ext = bp2_image_grid_thw.repeat(1, bp2_total_image_groups, 1)

qwen_cfg = bp2_model.qwen3_vl_with_expert.und_expert.config
bp2_vision_start_token_id = getattr(qwen_cfg, "vision_start_token_id", 151652)
bp2_vision_end_token_id = getattr(qwen_cfg, "vision_end_token_id", 151653)
bp2_image_token_id = qwen_cfg.image_token_id
bp2_spatial_merge_size = 2

# 只构造图像占位 tokens，不添加原始 task 文本 tokens。
bp2_token_rows = []
for b in range(bp2_image_grid_thw_ext.shape[0]):
    row = []
    for grid in bp2_image_grid_thw_ext[b]:
        num_img_token = int(torch.prod(grid).item() // (bp2_spatial_merge_size ** 2))
        row.extend([bp2_vision_start_token_id])
        row.extend([bp2_image_token_id] * num_img_token)
        row.extend([bp2_vision_end_token_id])
    bp2_token_rows.append(torch.tensor(row, dtype=torch.long, device=device))

bp2_max_token_len = max(row.numel() for row in bp2_token_rows)
bp2_lang_tokens_image_only = torch.full(
    (len(bp2_token_rows), bp2_max_token_len),
    fill_value=0,
    dtype=torch.long,
    device=device,
)
bp2_lang_masks_image_only = torch.zeros_like(bp2_lang_tokens_image_only)
for b, row in enumerate(bp2_token_rows):
    bp2_lang_tokens_image_only[b, : row.numel()] = row
    bp2_lang_masks_image_only[b, : row.numel()] = 1

show_bp2("bp2_pixel_values original", bp2_pixel_values)
show_bp2("bp2_pixel_values_ext", bp2_pixel_values_ext)
show_bp2("bp2_image_grid_thw original", bp2_image_grid_thw)
show_bp2("bp2_image_grid_thw_ext", bp2_image_grid_thw_ext)
show_bp2("bp2_lang_tokens_image_only", bp2_lang_tokens_image_only)
show_bp2("bp2_lang_masks_image_only", bp2_lang_masks_image_only)
print("原始 lang token 长度:", bp2_batch[f"{OBS_PREFIX}input_ids"].shape[1])
print("image-only lang token 长度:", bp2_lang_tokens_image_only.shape[1])
print("image token count:", int((bp2_lang_tokens_image_only == bp2_image_token_id).sum().item()))

========== BP 方案1修正版 / 1. 扩增 pixel_values 并重建无 task 的 image-only lang_tokens ==========
bp2_pixel_values original                          shape=(1, 768, 1536), dtype=torch.float32, device=cuda:0
bp2_pixel_values_ext                               shape=(1, 3840, 1536), dtype=torch.float32, device=cuda:0
bp2_image_grid_thw original                        shape=(1, 3, 3), dtype=torch.int64, device=cuda:0
bp2_image_grid_thw_ext                             shape=(1, 15, 3), dtype=torch.int64, device=cuda:0
bp2_lang_tokens_image_only                         shape=(1, 990), dtype=torch.int64, device=cuda:0
bp2_lang_masks_image_only                          shape=(1, 990), dtype=torch.int64, device=cuda:0
原始 lang token 长度: 246
image-only lang token 长度: 990
image token count: 960


In [45]:
print("========== BP 方案1修正版 / 2. 构造 4 块 action_state_emb ==========")

# bp2_actions: (B, 50, 32)
#   50 是 TBot 的 action chunk 时间长度，即模型一次监督/预测 50 步动作。
#   这里按 4 个 behavior 图像块把 50 步 action 分段聚合，让 action/state token 与图像块对齐。
# bp2_state: (B, 32)
#   当前时刻 proprio/state。实验里先复制到 4 个块中，与每块 action 摘要拼接。
B, action_horizon, action_dim = bp2_actions.shape
state_dim = bp2_state.shape[-1]
steps_per_behavior_chunk = (action_horizon + bp2_behavior_chunks - 1) // bp2_behavior_chunks
padded_horizon = steps_per_behavior_chunk * bp2_behavior_chunks
pad_steps = padded_horizon - action_horizon

if pad_steps > 0:
    bp2_actions_padded = F.pad(bp2_actions, (0, 0, 0, pad_steps))
else:
    bp2_actions_padded = bp2_actions

# (B, padded_horizon, action_dim) -> (B, 4, steps_per_chunk, action_dim)
bp2_actions_by_chunk = bp2_actions_padded.view(
    B,
    bp2_behavior_chunks,
    steps_per_behavior_chunk,
    action_dim,
)

# 每个 behavior 图像块对应一段 action 的均值摘要: (B, 4, action_dim)
bp2_action_chunk_summary = bp2_actions_by_chunk.mean(dim=2)

# 当前 state 复制到每个 behavior 块: (B, 4, state_dim)
bp2_state_by_chunk = bp2_state[:, None, :].expand(B, bp2_behavior_chunks, state_dim)

# 每块 prompt token 的输入: [state_chunk, action_chunk_summary]
# shape: (B, 4, state_dim + action_dim)
bp2_action_state_chunk = torch.cat([bp2_state_by_chunk, bp2_action_chunk_summary], dim=-1)

# 一个共享 MLP 将每个 action-state 块映射到 und_expert hidden dim: (B, 4, 2048)
bp2_action_state_mlp = nn.Sequential(
    nn.Linear(state_dim + action_dim, bp2_hidden_dim, device=device, dtype=torch.float32),
    nn.SiLU(),
    nn.Linear(bp2_hidden_dim, bp2_hidden_dim, device=device, dtype=torch.float32),
)
bp2_action_state_emb = bp2_action_state_mlp(bp2_action_state_chunk.to(torch.float32))

# 这就是要拼到 prefix_embs 前面的 learned behavior prompt tokens。
bp2_prompt_embs = bp2_action_state_emb
bp2_prompt_mask = torch.ones(
    bp2_prompt_embs.shape[:2],
    dtype=torch.bool,
    device=bp2_prompt_embs.device,
)
bp2_prompt_att_mask = torch.zeros_like(bp2_prompt_mask)

show_bp2("bp2_state", bp2_state, "当前 state，后续复制到 4 个 behavior 块")
show_bp2("bp2_actions", bp2_actions, "50 是 action chunk 的时间步数")
show_bp2("bp2_actions_padded", bp2_actions_padded, f"pad 到 {padded_horizon} 步，便于均分 4 块")
show_bp2("bp2_actions_by_chunk", bp2_actions_by_chunk, "按 4 个 behavior 图像块切分 action")
show_bp2("bp2_action_chunk_summary", bp2_action_chunk_summary, "每块 action 的均值摘要")
show_bp2("bp2_state_by_chunk", bp2_state_by_chunk, "同一当前 state 对齐到 4 个块")
show_bp2("bp2_action_state_chunk", bp2_action_state_chunk, "每块拼接 state + action summary")
show_bp2("bp2_action_state_emb", bp2_action_state_emb, "最终 action_state_emb，4 个 prefix prompt tokens")
show_bp2("bp2_prompt_embs", bp2_prompt_embs, "将拼到 prefix_embs 前面")
print("steps_per_behavior_chunk =", steps_per_behavior_chunk)
print("pad_steps =", pad_steps)

========== BP 方案1修正版 / 2. 构造 4 块 action_state_emb ==========
bp2_state                                          shape=(1, 32), dtype=torch.float32, device=cuda:0  # 当前 state，后续复制到 4 个 behavior 块
bp2_actions                                        shape=(1, 50, 32), dtype=torch.float32, device=cuda:0  # 50 是 action chunk 的时间步数
bp2_actions_padded                                 shape=(1, 52, 32), dtype=torch.float32, device=cuda:0  # pad 到 52 步，便于均分 4 块
bp2_actions_by_chunk                               shape=(1, 4, 13, 32), dtype=torch.float32, device=cuda:0  # 按 4 个 behavior 图像块切分 action
bp2_action_chunk_summary                           shape=(1, 4, 32), dtype=torch.float32, device=cuda:0  # 每块 action 的均值摘要
bp2_state_by_chunk                                 shape=(1, 4, 32), dtype=torch.float32, device=cuda:0  # 同一当前 state 对齐到 4 个块
bp2_action_state_chunk                             shape=(1, 4, 64), dtype=torch.float32, device=cuda:0  # 每块拼接 state + action summary
bp2_action_state_emb 

In [46]:
print("========== BP 方案1修正版 / 3. embed_prefix image-only + 拼接 action_state_emb ==========")

bp2_noise = bp2_model.sample_noise(bp2_actions.shape, bp2_actions.device)
bp2_time = bp2_model.sample_time(bp2_actions.shape[0], bp2_actions.device)
bp2_x_t = bp2_time[:, None, None] * bp2_noise + (1 - bp2_time[:, None, None]) * bp2_actions
bp2_u_t = bp2_noise - bp2_actions

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    bp2_prefix_embs_image_only, bp2_prefix_pad_masks_image_only, bp2_prefix_att_masks_image_only = bp2_model.embed_prefix(
        bp2_pixel_values_ext,
        bp2_image_grid_thw_ext,
        bp2_lang_tokens_image_only,
        bp2_lang_masks_image_only,
    )
    bp2_middle_embs, bp2_middle_pad_masks, bp2_middle_att_masks = bp2_model.embed_middle(
        bp2_images[:, :, :2],
        bp2_img_masks,
    )
    bp2_suffix_embs, bp2_suffix_pad_masks, bp2_suffix_att_masks = bp2_model.embed_suffix(
        bp2_state,
        bp2_x_t,
        bp2_time,
    )

bp2_prompt_embs = bp2_prompt_embs.to(dtype=bp2_prefix_embs_image_only.dtype)
bp2_prefix_embs_ext = torch.cat([bp2_prompt_embs, bp2_prefix_embs_image_only], dim=1)
bp2_prefix_pad_masks_ext = torch.cat([bp2_prompt_mask, bp2_prefix_pad_masks_image_only], dim=1)
bp2_prefix_att_masks_ext = torch.cat([bp2_prompt_att_mask, bp2_prefix_att_masks_image_only], dim=1)

bp2_pseudo_token_id = 777
bp2_prompt_token_ids = torch.full(
    bp2_prompt_mask.shape,
    bp2_pseudo_token_id,
    dtype=bp2_lang_tokens_image_only.dtype,
    device=bp2_lang_tokens_image_only.device,
)
bp2_lang_tokens_ext = torch.cat([bp2_prompt_token_ids, bp2_lang_tokens_image_only], dim=1)
bp2_lang_masks_ext = torch.cat([
    bp2_prompt_mask.to(bp2_lang_masks_image_only.dtype),
    bp2_lang_masks_image_only,
], dim=1)

if (
    bp2_model.qwen3_vl_with_expert.und_expert.language_model.layers[0].self_attn.q_proj.weight.dtype
    == torch.bfloat16
):
    bp2_prefix_embs_ext = bp2_prefix_embs_ext.to(dtype=torch.bfloat16)
    bp2_middle_embs = bp2_middle_embs.to(dtype=torch.bfloat16)
    bp2_suffix_embs = bp2_suffix_embs.to(dtype=torch.bfloat16)

bp2_pad_masks = torch.cat([bp2_prefix_pad_masks_ext, bp2_middle_pad_masks, bp2_suffix_pad_masks], dim=1)
bp2_att_masks = torch.cat([bp2_prefix_att_masks_ext, bp2_middle_att_masks, bp2_suffix_att_masks], dim=1)

bp2_att_2d_masks = bp2_model.build_training_attention_mask(
    bp2_pad_masks,
    bp2_att_masks,
    prefix_len=bp2_prefix_pad_masks_ext.shape[1],
)
bp2_position_ids, bp2_rope_deltas = bp2_model.get_position_ids(
    bp2_lang_tokens_ext,
    bp2_image_grid_thw_ext,
    bp2_pad_masks,
)
bp2_att_2d_masks_4d = bp2_model._prepare_attention_masks_4d(bp2_att_2d_masks)

show_bp2("bp2_prefix_embs_image_only", bp2_prefix_embs_image_only)
show_bp2("bp2_prefix_embs_ext", bp2_prefix_embs_ext)
show_bp2("bp2_lang_tokens_ext", bp2_lang_tokens_ext)
show_bp2("bp2_middle_embs", bp2_middle_embs)
show_bp2("bp2_suffix_embs", bp2_suffix_embs)
show_bp2("bp2_pad_masks", bp2_pad_masks)
show_bp2("bp2_position_ids", bp2_position_ids)
show_bp2("bp2_att_2d_masks_4d", bp2_att_2d_masks_4d)
print("bp2 prefix image-only/ext:", bp2_prefix_embs_image_only.shape[1], "->", bp2_prefix_embs_ext.shape[1])
print("bp2 total_len:", bp2_pad_masks.shape[1])

========== BP 方案1修正版 / 3. embed_prefix image-only + 拼接 action_state_emb ==========
bp2_prefix_embs_image_only                         shape=(1, 990, 2048), dtype=torch.bfloat16, device=cuda:0
bp2_prefix_embs_ext                                shape=(1, 994, 2048), dtype=torch.bfloat16, device=cuda:0
bp2_lang_tokens_ext                                shape=(1, 994), dtype=torch.int64, device=cuda:0
bp2_middle_embs                                    shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_suffix_embs                                    shape=(1, 51, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_pad_masks                                      shape=(1, 1573), dtype=torch.bool, device=cuda:0
bp2_position_ids                                   shape=(3, 1, 1573), dtype=torch.int64, device=cuda:0
bp2_att_2d_masks_4d                                shape=(1, 1, 1573, 1573), dtype=torch.float32, device=cuda:0
bp2 prefix image-only/ext: 990 -> 994
bp2 total_len: 1573


In [47]:
print("========== BP 方案1修正版 / 4. MoT forward + loss ==========")

bp2_collect_middle_layers = bp2_model.query_layer_indices if bp2_model.da3_teacher is not None else None
print("bp2_collect_middle_layers =", bp2_collect_middle_layers)

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    bp2_outputs = bp2_model.qwen3_vl_with_expert.forward(
        attention_mask=bp2_att_2d_masks_4d,
        position_ids=bp2_position_ids,
        past_key_values=None,
        inputs_embeds=[bp2_prefix_embs_ext, bp2_middle_embs, bp2_suffix_embs],
        use_cache=False,
        collect_middle_layers=bp2_collect_middle_layers,
    )

if bp2_collect_middle_layers is None:
    (_, bp2_middle_out, bp2_suffix_out), _ = bp2_outputs
    bp2_middle_layer_outputs = ()
else:
    (_, bp2_middle_out, bp2_suffix_out), _, bp2_middle_layer_outputs = bp2_outputs

show_bp2("bp2_middle_out", bp2_middle_out)
show_bp2("bp2_suffix_out", bp2_suffix_out)
for i, x in enumerate(bp2_middle_layer_outputs):
    show_bp2(f"bp2_middle_layer_outputs[{i}]", x)

if float(policy.config.lambda_gen) > 0.0:
    bp2_middle_visual_out, bp2_middle_query_out = bp2_model.split_middle_tokens(bp2_middle_out)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
        bp2_pred_cosmos_features = bp2_model.decode_cosmos(bp2_middle_visual_out.to(dtype=torch.float32))
        bp2_future_embs = bp2_model.get_cosmos_features(bp2_images[:, :, 2])
    bp2_loss_gen = F.mse_loss(
        bp2_pred_cosmos_features[bp2_img_masks],
        bp2_future_embs.to(dtype=torch.float32)[bp2_img_masks],
    )
else:
    bp2_middle_visual_out, bp2_middle_query_out = None, None
    bp2_loss_gen = bp2_middle_out.new_zeros((), dtype=torch.float32)

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    bp2_loss_3d, bp2_loss_3d_logs = bp2_model.compute_3d_query_loss(
        tuple(bp2_middle_layer_outputs),
        bp2_images[:, :, 2],
        bp2_img_masks,
    )

bp2_suffix_out_action = bp2_suffix_out[:, -policy.config.chunk_size :].to(dtype=torch.float32)
bp2_v_t = bp2_model.action_out_proj(bp2_suffix_out_action)
bp2_losses_action = F.mse_loss(bp2_u_t, bp2_v_t, reduction="none")

show_bp2("bp2_middle_visual_out", bp2_middle_visual_out)
show_bp2("bp2_suffix_out_action", bp2_suffix_out_action)
show_bp2("bp2_v_t", bp2_v_t)
show_bp2("bp2_losses_action raw", bp2_losses_action)
print("bp2_loss_gen =", bp2_loss_gen.item())
print("bp2_loss_3d =", bp2_loss_3d.item())

========== BP 方案1修正版 / 4. MoT forward + loss ==========
bp2_collect_middle_layers = (13, 19, 23, 27)
bp2_middle_out                                     shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_suffix_out                                     shape=(1, 51, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_middle_layer_outputs[0]                        shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_middle_layer_outputs[1]                        shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_middle_layer_outputs[2]                        shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_middle_layer_outputs[3]                        shape=(1, 528, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_middle_visual_out                              shape=(1, 96, 1024), dtype=torch.bfloat16, device=cuda:0
bp2_suffix_out_action                              shape=(1, 50, 1024), dtype=torch.float32, device=cuda:0
bp2_v_t                        

方案1修正版说明：这里的 `prefix_embs` 由两部分组成：前面是 `action_state_emb`，后面是 image-only Qwen prefix embeddings。`action_state_emb` 的 shape 是 `(B, 4, 2048)`，4 对应 4 个 behavior 图像块；每个 token 来自当前 state 与对应 action 分段摘要的拼接 MLP。image-only `lang_tokens` 只包含扩增后的图像占位 token，没有原始 task 文本 token。当前三路图像被复制 4 次作为 behavior prompt 图像，再追加原始三路当前图像，因此 `pixel_values` 和 `image_grid_thw` 都同步扩展了 5 倍。

In [48]:
print("========== BP 方案1修正版 / 5. 汇总 final loss ==========")

bp2_original_action_dim = policy.config.output_features[ACTION].shape[0]
bp2_losses_action_clipped = bp2_losses_action[:, :, :bp2_original_action_dim]

bp2_action_loss_mask = bp2_batch.get(SAMPLE_ACTION_LOSS_MASK)
if bp2_action_loss_mask is None:
    bp2_action_loss_mask = torch.ones(
        bp2_losses_action_clipped.shape[0],
        dtype=torch.bool,
        device=bp2_losses_action_clipped.device,
    )
else:
    bp2_action_loss_mask = bp2_action_loss_mask.to(bp2_losses_action_clipped.device)
    if bp2_action_loss_mask.ndim > 1:
        bp2_action_loss_mask = bp2_action_loss_mask.squeeze(-1)
    bp2_action_loss_mask = bp2_action_loss_mask > 0.5

if bp2_action_loss_mask.any():
    bp2_sample_losses_action = bp2_losses_action_clipped[bp2_action_loss_mask]
    if policy.config.mask_action_dim_padding_loss:
        bp2_valid_action_dim = min(int(policy.config.action_loss_valid_dim), bp2_original_action_dim)
        bp2_loss_action = bp2_sample_losses_action[:, :, :bp2_valid_action_dim].mean()
    else:
        bp2_loss_action = bp2_sample_losses_action.mean()
else:
    bp2_loss_action = bp2_losses_action_clipped.new_zeros(())

bp2_loss = bp2_loss_action + policy.config.lambda_gen * bp2_loss_gen + policy.config.lambda_3d * bp2_loss_3d

bp2_loss_dict = {
    "loss": bp2_loss.item(),
    "loss_action": bp2_loss_action.item(),
    "loss_gen": bp2_loss_gen.item(),
    "loss_3d": bp2_loss_3d.item(),
}
for key, value in bp2_loss_3d_logs.items():
    bp2_loss_dict[key] = float(value.item())

show_bp2("bp2_losses_action_clipped", bp2_losses_action_clipped)
show_bp2("bp2_action_loss_mask", bp2_action_loss_mask)
print("bp2_loss =", bp2_loss.item())
print("bp2_loss_action =", bp2_loss_action.item())
print("bp2_loss_gen =", bp2_loss_gen.item())
print("bp2_loss_3d =", bp2_loss_3d.item())
bp2_loss_dict

========== BP 方案1修正版 / 5. 汇总 final loss ==========
bp2_losses_action_clipped                          shape=(1, 50, 32), dtype=torch.float32, device=cuda:0
bp2_action_loss_mask                               shape=(1,), dtype=torch.bool, device=cuda:0
bp2_loss = 0.2706314027309418
bp2_loss_action = 0.24696628749370575
bp2_loss_gen = 1.976219892501831
bp2_loss_3d = 0.39029133319854736


{'loss': 0.2706314027309418,
 'loss_action': 0.24696628749370575,
 'loss_gen': 1.976219892501831,
 'loss_3d': 0.39029133319854736,
 'time_3d_teacher_forward_s': 0.044540125876665115,
 'loss_3d_q13_t11': 0.11133645474910736,
 'loss_3d_q19_t15': 0.2569740116596222,
 'loss_3d_q23_t19': 0.469971239566803,
 'loss_3d_q27_t23': 0.7228837013244629}

  00. BPPadOrSampleChunksFn
  01. BPResizeImagesWithPadFn
  02. BPRemapImageKeyTransformFn
  03. BPNormalizeTransformFn
  04. BPComposeFieldsTransform
  05. BPPadStateAndActionTransformFn
  06. BPImgOnlyQwen3VLTransformFn
  07. InjectMissingStateActionTransformFn # 这个舍弃目前不需要就不用
  08. ResizeImagesWithPadFn
  09. RemapImageKeyTransformFn
  10. NormalizeTransformFn
  11. ComposeFieldsTransform # 这个也舍弃目前不需要就不用
  12. PadStateAndActionTransformFn
  13. CurrentImageOnlyQwenTransformFn
  14. UnifyTBotSA1BPInputsTransformFn